# Week 3, day 2 — Worksheet 04 SOLUTIONS: stack   (L04)

Executed in the lab image (pandas 3.0.5) against the real files in `data/`.
Every quoted number is what it actually printed.

Questions 6 and 10 are the ones to re-read. `stack()`'s handling of missing
values was rewritten, and the old argument for controlling it now raises.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 04 — stack. Run this once.
import pandas as pd

# The spreadsheet-style export: Technology sales, regions down, YEARS across.
wide = pd.read_csv("data/technology_wide.csv")

orders = pd.read_csv("data/orders_long.csv")

print(wide.to_string(index=False))
print()
print("columns:", list(wide.columns))

PART A — rescuing a wide export

### Question 1

A **Series** of `32` rows with index names `['Region', None]`.

The columns folded down into a second row level, so a 8x4 grid became 32
rows. `stack()` returns a Series here because the frame had a single set of
columns and all of them were stacked.

The new level is **unnamed** — `None`. That is because the wide file's
column headings were just labels; nothing recorded that they were years.
That missing name is what Q2 fixes, and it matters: `unstack("Year")` cannot
work on a level with no name.

In [ ]:
tidy = wide.set_index("Region").stack()
print(tidy.head(8).to_string())
print()
print("type:  ", type(tidy).__name__)
print("length:", len(tidy))
print("index names:", tidy.index.names)

### Question 2

A `(32, 3)` frame with columns `['Region', 'Year', 'Sales']`. -> `Year` dtype is **`str`**.

Now it is tidy long format: one observation per row, and `Year` is data
rather than a column heading.

But `Year` came through as **text**, because it was a CSV column *name* and
names are strings. Sorting it works by luck (four-digit years sort the same
as text and as numbers, the same coincidence as ISO dates in the week 3, day 1
class), and arithmetic on it does not. `.astype(int)` at this point is worth
the keystrokes.

In [ ]:
tidy = (wide.set_index("Region")
             .stack()
             .rename_axis(["Region", "Year"])
             .reset_index(name="Sales"))
print(tidy.head(6).to_string(index=False))
print()
print("shape:", tidy.shape)
print("columns:", list(tidy.columns))
print("Year dtype:", tidy["Year"].dtype)

### Question 3

Per-year totals from the stacked frame match the same totals computed from `orders`. -> `Year` values come out as `['2009', '2010', '2011', '2012']` — strings.

The `groupby` is the proof that the reshape worked: you could not write
`groupby("Year")` against the wide frame at all, because there was no `Year`
column to group on.

That is the practical test of tidiness. If answering 'per X' requires
naming columns one at a time, X is trapped in your schema and wants
stacking out of it.

In [ ]:
tidy = (wide.set_index("Region").stack()
             .rename_axis(["Region", "Year"]).reset_index(name="Sales"))
from_wide = tidy.groupby("Year")["Sales"].sum().round(2)
print(from_wide.to_string())
print()
tech = orders[orders["Category"] == "Technology"]
from_long = tech.groupby("Year")["Sales"].sum().round(2)
print(from_long.to_string())
print()
print("Year values from the stacked frame:", list(tidy["Year"].unique()))

### Question 4

`32` cells in, **`32`** rows out, difference `0` — and the grid contained **2** blank cells.

The two blanks survived as rows holding `NaN`. Nothing was dropped.

That is the pandas 3 behaviour and it is the opposite of what most
tutorials describe: `stack()` used to discard missing combinations by
default, so this would have printed 30. If you follow older material and
expect 30, you will conclude your data is wrong when it is your Pandas that
changed.

In [ ]:
tidy = wide.set_index("Region").stack()
cells = wide.set_index("Region").size
print("cells in the wide grid:", cells)
print("rows after stacking:   ", len(tidy))
print("difference:            ", cells - len(tidy))
print()
print("blank cells in the wide grid:",
      int(wide.set_index("Region").isna().sum().sum()))

PART B — the round trip

### Question 5

Complete data: `32` -> `32`, `.equals()` **`True`**.

Lossless, because the grid was full. `unstack` then `stack` is the identity
when every row-column combination exists.

In [ ]:
base = orders.groupby(["Region", "Year"])["Sales"].sum().round(2)
rt = base.unstack().stack()
print("original length:", len(base))
print("round trip:     ", len(rt))
print("equal:", rt.equals(base))
print()
print(rt.head(4).to_string())

### Question 6

`30` combinations -> `32` cells (2 `NaN`) -> `32` rows after `stack()`, and `.equals()` is **`False`**. -> `2` `NaN` rows kept.

The round trip is **not** lossless here: you started with 30 rows and got
32 back. The two extra rows are the combinations `unstack` invented, and
`stack` faithfully preserved them.

So the reshape added two rows to your data that no order ever produced.
Group or count that result and Nunavut now appears to have Technology
records in 2009 — with a missing value, but present as rows.

This is why the deck's slide says stack/unstack are mirrors 'mostly'. They
are exact mirrors on a complete grid and lossy in both directions on an
incomplete one.

In [ ]:
tech = orders[orders["Category"] == "Technology"]
base = tech.groupby(["Region", "Year"])["Sales"].sum().round(2)
wide_t = base.unstack()

print("original combinations:", len(base))
print("cells after unstack:  ", wide_t.size)
print("NaN cells:            ", int(wide_t.isna().sum().sum()))
print()
rt = wide_t.stack()
print("rows after stack():   ", len(rt))
print("equal to the original:", rt.equals(base))
print()
print("NaN rows kept by stack():", int(rt.isna().sum()))

### Question 7

The two `NaN` rows are `Nunavut 2009` and `Nunavut 2012`.

Named, so you can decide what to do with them rather than discovering them
later as an odd count. Printing the `NaN` rows after any reshape is a
cheap habit — `s[s.isna()]` on a Series, `df[df.isna().any(axis=1)]` on a
frame.

In [ ]:
tech = orders[orders["Category"] == "Technology"]
rt = tech.groupby(["Region", "Year"])["Sales"].sum().round(2).unstack().stack()
print(rt[rt.isna()].to_string())

### Question 8

`.dropna()` -> `30` rows, `.equals()` the original **`True`**.

Dropping them explicitly restores the original exactly. The point is that
you did it *deliberately*, in a line someone can read, rather than relying
on a default that changed between versions.

That is the whole answer to the `dropna=` removal in Q10: the argument was
taken away because the behaviour is clearer as a separate step.

In [ ]:
tech = orders[orders["Category"] == "Technology"]
base = tech.groupby(["Region", "Year"])["Sales"].sum().round(2)
rt = base.unstack().stack().dropna()
print("length:", len(rt), "vs original", len(base))
print("equal:", rt.equals(base))

PART C — stacking a two-level column index

### Question 9

`(8, 12)` with 2 column levels -> after `stack("Category")`, `(24, 4)` with **1** column level and row index `['Region', 'Category']`.

One level moved from the columns to the rows, so the frame got taller (8 ->
24) and narrower (12 -> 4).

That is the whole of reshaping in one line: the data is a fixed number of
facts, and `stack`/`unstack` only decide how many of the identifying
variables live on each axis. 8x12 and 24x4 are the same 96 cells.

In [ ]:
three = orders.groupby(["Region", "Year", "Category"])["Sales"].sum().round(2)
two_col = three.unstack(["Year", "Category"])
print("two column levels:", two_col.shape, "| levels:", two_col.columns.nlevels)
print()
back = two_col.stack("Category")
print("after stack('Category'):", back.shape, "| column levels:", back.columns.nlevels)
print("row index names:", back.index.names)
print()
print(back.head(4).to_string())

### Question 10

`stack(dropna=False)` -> **raises** `ValueError: dropna must be unspecified as the new implementation does not introduce rows of NA values. This argument will be removed in a future version of pandas.`

An argument that worked for years, appears throughout the documentation
and tutorials online, and is now an error.

The message explains the reasoning: the rewritten `stack()` never *drops*
NA rows, so there is nothing for `dropna` to control. The old default was
`dropna=True`, which quietly discarded exactly the rows Q7 printed.

Two things to take from this. First, the behaviour change is silent for
anyone who did not pass the argument — their round trips return more rows
than they used to, with no error. Second, when a deck, a tutorial and your
interpreter disagree, the interpreter is the one shipping to production.
Run the line.

In [ ]:
print(wide.set_index("Region").stack(dropna=False))